# 05 — Clustering and Pattern Segmentation

Scale all 30 features, evaluate K-means for K=2–6, choose K=3 for the requested cluster profiling, and use hierarchical clustering as a cross-check.

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from scipy.cluster.hierarchy import linkage,dendrogram
FEATURE_BASE = ['radius','texture','perimeter','area','smoothness','compactness','concavity','concave_points','symmetry','fractal_dimension']
FEATURES = [f'{stat}_{feat}' for stat in ['mean','se','worst'] for feat in FEATURE_BASE]
df=pd.read_csv('data/wdbc.data',header=None,names=['id','diagnosis']+FEATURES)
X=df[FEATURES]; y=(df.diagnosis=='M').astype(int)
scaler=StandardScaler(); Xs=scaler.fit_transform(X)

In [ ]:
scores={}; inertias={}
for k in range(2,7):
 km=KMeans(n_clusters=k,random_state=42,n_init=20).fit(Xs)
 inertias[k]=km.inertia_; scores[k]=silhouette_score(Xs,km.labels_)
print('Silhouette:',scores)

In [ ]:
plt.plot(list(inertias),list(inertias.values()),marker='o')
plt.xticks(list(inertias.keys())); plt.xlabel('K'); plt.ylabel('Inertia'); plt.title('K-means Elbow Plot'); plt.tight_layout(); plt.show()

In [ ]:
k=3
km=KMeans(n_clusters=k,random_state=42,n_init=20).fit(Xs)
df['cluster']=km.labels_+1
for c,g in df.groupby('cluster'):
 print('Cluster',c,'size',len(g),'M',sum(g.diagnosis=='M'),'B',sum(g.diagnosis=='B'),'M%',round(100*sum(g.diagnosis=='M')/len(g),2))

In [ ]:
Z=linkage(Xs,method='ward')
plt.figure(figsize=(12,5)); dendrogram(Z,no_labels=True); plt.title('Hierarchical Clustering Dendrogram (Ward)'); plt.xlabel('Observations'); plt.ylabel('Distance'); plt.tight_layout(); plt.show()

In [ ]:
for c in sorted(df.cluster.unique()):
 g=df[df.cluster==c]
 means=g[FEATURES].mean(); overall=X.mean()
 diff=(means-overall).abs().sort_values(ascending=False).head(6)
 print('Cluster',c,'top distinguishing features:',list(diff.index))